# Resource URL verification

Checks every `resource_list_url` in a spreadsheet for a real HTTP 200 and for "soft 404s" —
pages that return 200 but whose title/heading text says the content isn't actually there. See
`../SKILL.md` for the full rationale. Writes a **new** output file; never overwrites the input.

In [ ]:
import re
import pathlib
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Typically the de-duplicated output of institution-resource-list-cleanup, so each URL is only
# checked once regardless of how many institutions share it.
INPUT_PATH = pathlib.Path(
    "../../output/accredited_nonprofit_secular_institutions_unique_resource_urls.xlsx"
)

MAX_WORKERS = 15
TIMEOUT_SECONDS = 15
MAX_BODY_BYTES = 40_000  # plenty for <title>/<h1-3> tags, which sit near the top of the document

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

TITLE_RE = re.compile(r"<title[^>]*>(.*?)</title>", re.IGNORECASE | re.DOTALL)
HEADING_RE = re.compile(r"<h[1-3][^>]*>(.*?)</h[1-3]>", re.IGNORECASE | re.DOTALL)
TAG_RE = re.compile(r"<[^>]+>")

# Checked only against <title>/<h1-3> text (not full page body) so this targets the "large text
# told the user it's missing" signal specifically, without false-positiving on unrelated body
# copy that happens to mention an HTTP error code somewhere on an otherwise-real page.
SOFT_404_PATTERNS = re.compile(
    r"\b404\b|\b400\b|\b410\b|error\s*404|404\s*error|"
    r"page not found|page cannot be found|page could not be found|"
    r"we can.?t find (?:that|this) page|we couldn.?t find (?:that|this) page|"
    r"the page you.{0,3}re looking for|the page you requested|"
    r"this page (?:does not|doesn.?t) exist|no longer exists|"
    r"content not found|resource not found|file not found|"
    r"oops.{0,10}(?:page|that)|nothing (?:was )?found here|"
    r"broken link|link is broken|page has (?:been )?(?:removed|moved)|"
    r"sorry.{0,15}(?:can.?t find|not found)|bad request|access forbidden",
    re.IGNORECASE,
)


def make_session(max_workers, connect_retries=1, read_retries=1, status_retries=2,
                  status_forcelist=(502, 503, 504), backoff_factor=1):
    # connect_retries/read_retries deliberately small: a host that's refusing connections or not
    # responding at all is usually just dead, and retrying that repeatedly with escalating backoff
    # burns a lot of wall-clock time for no benefit. status_retries (for explicit 429/5xx HTTP
    # responses) is where the useful signal is -- those genuinely mean "try again later."
    session = requests.Session()
    retry = Retry(
        total=max(connect_retries, read_retries, status_retries),
        connect=connect_retries,
        read=read_retries,
        status=status_retries,
        backoff_factor=backoff_factor,
        status_forcelist=list(status_forcelist),
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=max_workers, pool_connections=max_workers)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def strip_tags(fragment):
    text = TAG_RE.sub(" ", fragment)
    text = re.sub(r"&nbsp;|&amp;|&mdash;|&ndash;|&#\d+;", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_title_and_headings(html):
    title_match = TITLE_RE.search(html)
    title = strip_tags(title_match.group(1)) if title_match else ""
    headings = [strip_tags(h) for h in HEADING_RE.findall(html)]
    return title, headings


def check_one_url(session, url, timeout=TIMEOUT_SECONDS):
    result = {
        "resource_list_url": url,
        "http_status": None,
        "final_url": "",
        "is_soft_404": False,
        "soft_404_match_text": "",
        "page_title": "",
    }
    if not isinstance(url, str) or not url.strip():
        result["http_status"] = "NoURL"
        return result
    try:
        resp = session.get(
            url, headers=HEADERS, timeout=timeout, allow_redirects=True, stream=True
        )
    except requests.exceptions.SSLError:
        result["http_status"] = "SSLError"
        return result
    except requests.exceptions.Timeout:
        result["http_status"] = "Timeout"
        return result
    except requests.exceptions.TooManyRedirects:
        result["http_status"] = "TooManyRedirects"
        return result
    except requests.exceptions.ConnectionError:
        result["http_status"] = "ConnectionError"
        return result
    except requests.exceptions.RequestException as exc:
        result["http_status"] = type(exc).__name__
        return result

    with resp:
        result["http_status"] = resp.status_code
        result["final_url"] = resp.url
        content_type = resp.headers.get("Content-Type", "")
        if resp.status_code == 200 and "html" in content_type.lower():
            body = ""
            try:
                chunks = []
                total = 0
                for chunk in resp.iter_content(chunk_size=8192):
                    if not chunk:
                        break
                    chunks.append(chunk)
                    total += len(chunk)
                    if total >= MAX_BODY_BYTES:
                        break
                raw = b"".join(chunks)
                body = raw.decode(resp.encoding or "utf-8", errors="replace")
            except Exception:
                body = ""
            if body:
                title, headings = extract_title_and_headings(body)
                result["page_title"] = title
                combined = " | ".join([title] + headings)
                match = SOFT_404_PATTERNS.search(combined)
                if match:
                    result["is_soft_404"] = True
                    result["soft_404_match_text"] = combined[:300]
    return result


def build_check_notes(row):
    status = row["http_status"]
    if isinstance(status, str):
        return f"Request failed: {status}"
    if status == 200:
        if row["is_soft_404"]:
            return f"HTTP 200 but looks like a soft error page (matched: '{row['soft_404_match_text']}')"
        return "OK"
    return f"HTTP {status}"

In [ ]:
df = pd.read_excel(INPUT_PATH)

if df["resource_list_url"].duplicated().any():
    n_dupes = int(df["resource_list_url"].duplicated().sum())
    print(f"Note: {n_dupes} duplicate resource_list_url values in the input "
          f"(each will still only be requested once below, results applied to all matching rows).")

unique_urls = df["resource_list_url"].dropna().unique().tolist()
print(f"Checking {len(unique_urls)} unique URLs with {MAX_WORKERS} workers...")

session = make_session(MAX_WORKERS)
results_by_url = {}
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(check_one_url, session, url, TIMEOUT_SECONDS): url for url in unique_urls}
    for i, future in enumerate(as_completed(futures), start=1):
        url = futures[future]
        try:
            results_by_url[url] = future.result()
        except Exception as exc:
            results_by_url[url] = {
                "resource_list_url": url, "http_status": type(exc).__name__,
                "final_url": "", "is_soft_404": False, "soft_404_match_text": "", "page_title": "",
            }
        if i % 100 == 0 or i == len(unique_urls):
            elapsed = time.time() - start
            print(f"  {i}/{len(unique_urls)} checked ({elapsed:.0f}s elapsed)")

checked_at = datetime.now(timezone.utc).isoformat()
results_df = pd.DataFrame(results_by_url.values())
results_df["checked_at"] = checked_at

output = df.merge(results_df, on="resource_list_url", how="left")
output["check_notes"] = output.apply(build_check_notes, axis=1)

OUTPUT_PATH = INPUT_PATH.with_name("unique_resources_checked.xlsx")
output.to_excel(OUTPUT_PATH, index=False)

n_ok = int(((output["http_status"] == 200) & (~output["is_soft_404"])).sum())
n_soft_404 = int(((output["http_status"] == 200) & (output["is_soft_404"])).sum())
n_bad_status = int((output["http_status"].apply(lambda s: isinstance(s, (int,)) and s != 200)).sum())
n_failed = int((output["http_status"].apply(lambda s: isinstance(s, str))).sum())

print()
print(f"Input:  {len(df)} rows ({INPUT_PATH})")
print(f"Output: {len(output)} rows ({OUTPUT_PATH})")
print(f"  {n_ok} OK (HTTP 200, no soft-404 signal)")
print(f"  {n_soft_404} HTTP 200 but flagged as a possible soft 404")
print(f"  {n_bad_status} non-200 HTTP status")
print(f"  {n_failed} request failed (timeout/connection/SSL/other error)")

## Follow-up pass: recheck non-200 rows at lower concurrency

The first pass above uses 15 concurrent workers across ~1,800 different hosts. Some of the
non-200 results from a run like that are genuinely broken links — but some (particularly 429s, and
some `ConnectionError`s) can just be an anti-bot/rate-limit defense reacting to many
near-simultaneous requests, not a real dead link. Re-requesting only the failed subset, slower and
with fewer concurrent connections, resolves that ambiguity without having to re-check everything
that already came back clean.

This section:

- Reads a previously-checked file (`RECHECK_INPUT_PATH`, default: the output of the section above).
- Selects only rows where `http_status` isn't `200` (soft-404s are excluded — those already got a
  real 200 response and readable content confirming they're dead, so re-requesting them can't
  change that verdict).
- Re-requests just those URLs with far lower concurrency (`RECHECK_MAX_WORKERS`, default 3), a
  longer timeout, and a retry policy that also backs off on 429 specifically (in addition to
  502/503/504), since 429 means "you're being rate-limited, slow down and try again."
- Overwrites `http_status`/`final_url`/`is_soft_404`/`soft_404_match_text`/`page_title`/`checked_at`
  for just the rechecked rows, and adds `was_rechecked` and `http_status_before_recheck` so it's
  visible which rows were touched and what changed.
- Writes `unique_resources_double_checked.xlsx` — never overwrites the input.

In [ ]:
RECHECK_INPUT_PATH = OUTPUT_PATH  # defaults to this notebook's own first-pass output
RECHECK_MAX_WORKERS = 6
RECHECK_TIMEOUT_SECONDS = 12

recheck_df = pd.read_excel(RECHECK_INPUT_PATH)

needs_recheck = recheck_df["http_status"] != 200
to_recheck = recheck_df.loc[needs_recheck, "resource_list_url"].dropna().unique().tolist()
print(f"Rechecking {len(to_recheck)} URLs (non-200 on the first pass) "
      f"with {RECHECK_MAX_WORKERS} workers...")

# 429 added to the retry status list here (on top of 502/503/504) since a 429 during the recheck
# itself means "back off and try again" -- worth honoring automatically (respect_retry_after_header
# in make_session() honors a Retry-After header when the server sends one). connect/read retries
# stay small (see make_session's docstring comment) so genuinely dead hosts fail fast rather than
# burning minutes each on repeated connection attempts.
recheck_session = make_session(
    RECHECK_MAX_WORKERS,
    connect_retries=1, read_retries=1, status_retries=3,
    status_forcelist=(429, 502, 503, 504), backoff_factor=1.5,
)

recheck_results_by_url = {}
start = time.time()
with ThreadPoolExecutor(max_workers=RECHECK_MAX_WORKERS) as pool:
    futures = {
        pool.submit(check_one_url, recheck_session, url, RECHECK_TIMEOUT_SECONDS): url
        for url in to_recheck
    }
    for i, future in enumerate(as_completed(futures), start=1):
        url = futures[future]
        try:
            recheck_results_by_url[url] = future.result()
        except Exception as exc:
            recheck_results_by_url[url] = {
                "resource_list_url": url, "http_status": type(exc).__name__,
                "final_url": "", "is_soft_404": False, "soft_404_match_text": "", "page_title": "",
            }
        if i % 25 == 0 or i == len(to_recheck):
            print(f"  {i}/{len(to_recheck)} rechecked ({time.time() - start:.0f}s elapsed)")

recheck_checked_at = datetime.now(timezone.utc).isoformat()

recheck_df["was_rechecked"] = False
recheck_df["http_status_before_recheck"] = pd.NA

for idx, row in recheck_df.loc[needs_recheck].iterrows():
    res = recheck_results_by_url.get(row["resource_list_url"])
    if res is None:
        continue
    recheck_df.at[idx, "http_status_before_recheck"] = row["http_status"]
    recheck_df.at[idx, "was_rechecked"] = True
    recheck_df.at[idx, "http_status"] = res["http_status"]
    recheck_df.at[idx, "final_url"] = res["final_url"]
    recheck_df.at[idx, "is_soft_404"] = res["is_soft_404"]
    recheck_df.at[idx, "soft_404_match_text"] = res["soft_404_match_text"]
    recheck_df.at[idx, "page_title"] = res["page_title"]
    recheck_df.at[idx, "checked_at"] = recheck_checked_at

recheck_df["check_notes"] = recheck_df.apply(build_check_notes, axis=1)

RECHECK_OUTPUT_PATH = RECHECK_INPUT_PATH.with_name("unique_resources_double_checked.xlsx")
recheck_df.to_excel(RECHECK_OUTPUT_PATH, index=False)

n_now_ok = int((recheck_df["was_rechecked"] & (recheck_df["http_status"] == 200) & (~recheck_df["is_soft_404"])).sum())
n_still_bad = int((recheck_df["was_rechecked"] & ~((recheck_df["http_status"] == 200) & (~recheck_df["is_soft_404"]))).sum())

print()
print(f"Input:  {len(recheck_df)} rows ({RECHECK_INPUT_PATH})")
print(f"Output: {len(recheck_df)} rows ({RECHECK_OUTPUT_PATH})")
print(f"  {len(to_recheck)} rows rechecked")
print(f"  {n_now_ok} flipped to a clean 200 on recheck (likely rate-limiting/transient on the first pass)")
print(f"  {n_still_bad} still non-200 or soft-404 on recheck (more likely genuinely broken)")